# Run: play a game on the real robot (or in Gazebo)

**Milestone:** "Complete a full game with no human intervention" (README, proposed: 10 games).
Wires the real vision source and arm driver into `GameManager`, logs to `logging.dir`.

**TODO checklist**
- [ ] `RealVision`: adapter around gambit_vision (camera -> board assembly -> BoardObservation).
- [ ] `RealArm`: adapter around gambit_arm pick/place (or gambit_sim teleport in Gazebo).
- [ ] Choose the robot policy: `EnginePolicy` (strength-limited) or the teacher.
- [ ] Record per-move success / retries for the README measurement logs.

In [ ]:
%run ./game_loop.ipynb

In [ ]:
from pathlib import Path

REPO = Path("../..").resolve()
LOG_DIR = REPO / GAME_CFG["logging"]["dir"]
HUMAN_COLOR = chess.WHITE if GAME_CFG["game"]["human_color"] == "white" else chess.BLACK
USE_TEACHER = False          # True: gambit_teach TeacherPolicy (needs a player model)

## Adapters (TODO)

In [ ]:
class RealVision:
    """VisionSource on the real camera. TODO: call gambit_vision's board reader (capture frame,
    rectify with the ArUco homography, classify 64 squares) and return a BoardObservation."""

    def read(self) -> BoardObservation:
        raise NotImplementedError("RealVision.read")


class RealArm:
    """ArmInterface on the SO-ARM101 (gambit_arm) or Gazebo (gambit_sim teleport).
    TODO: forward each primitive to the arm package and return its success flag."""

    def pick(self, square): raise NotImplementedError
    def place(self, square): raise NotImplementedError
    def move_piece(self, from_sq, to_sq): raise NotImplementedError
    def capture_to_graveyard(self, square): raise NotImplementedError
    def place_from_graveyard(self, piece, square): raise NotImplementedError
    def park(self): raise NotImplementedError

## Play

In [ ]:
engine = make_engine(prefer_uci=True, elo=GAME_CFG["engine"]["default_elo"])
analyst = make_engine(prefer_uci=True)           # full strength, for logging evals
if USE_TEACHER:
    # %run ../../gambit_teach/notebooks/move_selection.ipynb  then build TeacherPolicy(engine, model)
    raise NotImplementedError("teacher wiring")
policy = EnginePolicy(engine)

logger = GameLogger(LOG_DIR)
tracker = HesitationTracker()
gm = GameManager(RealVision(), RealArm(), policy, HUMAN_COLOR,
                 listeners=[tracker, make_log_listener(logger, analyst)])
try:
    board = gm.run()
finally:
    logger.finish(gm.board.result(claim_draw=True) if gm.board.is_game_over(claim_draw=True) else "*")
    engine.close(); analyst.close()
print(gm.board.result(), logger.pgn_path)